#  Gold Layer - Date Dimension

The Date Dimension provides a centralized calendar table for reporting and analytics.

Instead of repeatedly extracting year, month, weekday, and quarter in every query or dashboard, these attributes are stored once in a reusable dimension table.

This improves report performance, simplifies Power BI development, and follows dimensional modeling best practices.

### Source

Silver Layer (`taxi.silver.yellow_taxi`)

### Target

`taxi.gold.dim_date`

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("taxi.silver.yellow_taxi")

In [0]:
dim_date = (
    silver_df
    .select("pickup_date")
    .distinct()
    .withColumn(
        "date_key",
        F.date_format("pickup_date", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "year",
        F.year("pickup_date")
    )
    .withColumn(
        "quarter",
        F.quarter("pickup_date")
    )
    .withColumn(
        "month",
        F.month("pickup_date")
    )
    .withColumn(
        "month_name",
        F.date_format("pickup_date", "MMMM")
    )
    .withColumn(
        "week",
        F.weekofyear("pickup_date")
    )
    .withColumn(
        "day",
        F.dayofmonth("pickup_date")
    )
    .withColumn(
        "day_name",
        F.date_format("pickup_date", "EEEE")
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("pickup_date").isin(1,7)
    )
)

In [0]:
display(dim_date)

In [0]:
(
    dim_date.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("taxi.gold.dim_date")
)

In [0]:
%sql
DESCRIBE DETAIL taxi.gold.dim_date;